In [ ]:
import requests
import gspread
import time
import sys
from oauth2client.service_account import ServiceAccountCredentials
from gspread.exceptions import APIError, WorksheetNotFound

# --- 1. CONFIGURATION ---
AUTH_URL = "https://dev.gnteq.app/api/identity/Authentication/GetToken"
BULK_TRACK_URL = "https://dev.gnteq.app/api/gnconnect/Tracking/BulkTracking"

USER_NAME = "NaqelCustomer"
PASSWORD = "n%A5E1Cust6mer"
CUSTOMER_CODE = "NL123456"
BRANCH_CODE = "NL567899"

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1hClSA4u_gE5KUudt2Vz2dHJVmh9a3AciCjycjB3Rvds/edit"
TARGETSHEET_URL = "https://docs.google.com/spreadsheets/d/1S6WaI1rXuf5ogN44ph8weA_z_bMCg85U3uaZ7RHvpfM/edit"
JSON_KEYFILE = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'

def get_bearer_token():
    payload = {"userName": USER_NAME, "password": PASSWORD}
    try:
        response = requests.post(AUTH_URL, json=payload)
        response.raise_for_status()
        return response.json().get("token")
    except Exception as e:
        print(f"❌ Naqel Auth Error: {e}")
        return None

def fetch_bulk_tracking(awbs, token):
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json"
    }
    payload = {
        "airwaybills": awbs,
        "customerCode": CUSTOMER_CODE,
        "branchCode": BRANCH_CODE
    }
    try:
        response = requests.post(BULK_TRACK_URL, json=payload, headers=headers)
        return response.json() if response.status_code == 200 else None
    except Exception as e:
        print(f"❌ Naqel API Error: {e}")
        return None

def safe_google_call(func, *args, **kwargs):
    """Retries Google Sheets calls if quota is exceeded."""
    for attempt in range(5):
        try:
            return func(*args, **kwargs)
        except APIError as e:
            if "429" in str(e):
                wait_time = (2 ** attempt) + 10
                print(f"⚠️ Quota hit. Waiting {wait_time}s before retry {attempt+1}/5...")
                time.sleep(wait_time)
            else:
                raise e
    raise Exception("Max retries exceeded for Google Sheets API")

def main():
    try:
        scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
        creds = ServiceAccountCredentials.from_json_keyfile_name(JSON_KEYFILE, scope)
        client = gspread.authorize(creds)
        
        print("Connecting to Google Sheets...")
        source_doc = safe_google_call(client.open_by_url, SPREADSHEET_URL)
        target_doc = safe_google_call(client.open_by_url, TARGETSHEET_URL)
        
        input_sheet = source_doc.worksheet("NAQEL_TRACK_ID")
        try:
            output_sheet = target_doc.worksheet("NAQEL_STATUS")
        except WorksheetNotFound:
            output_sheet = target_doc.add_worksheet(title="NAQEL_STATUS", rows="1000", cols="10")

        # Read Data
        all_data = safe_google_call(input_sheet.get, "A2:A")
        naqel_ids = [str(row[0]).strip() for row in all_data if row and str(row[0]).strip()]
        
        if not naqel_ids:
            print("No IDs found.")
            return

        token = get_bearer_token()
        if not token: return

        # Process Naqel API (Not subject to Google Quota)
        all_results = []
        CHUNK_SIZE = 15
        for i in range(0, len(naqel_ids), CHUNK_SIZE):
            chunk = naqel_ids[i:i + CHUNK_SIZE]
            print(f"Tracking Chunk {i//CHUNK_SIZE + 1}...")
            batch_data = fetch_bulk_tracking(chunk, token)
            if batch_data:
                all_results.extend(batch_data)
            time.sleep(1)

        if all_results:
            rows_to_write = []
            current_time = time.strftime("%Y-%m-%d %H:%M:%S")
            for item in all_results:
                rows_to_write.append([
                    item.get("airwaybillNumber", "N/A"),
                    item.get("lastStatus", "N/A"),
                    item.get("lastUpdateDate", "N/A"),
                    item.get("origin", "N/A"),
                    item.get("destination", "N/A"),
                    current_time
                ])
            
            # Write Data
            headers = [['Tracking ID', 'Status', 'Last Event', 'Origin', 'Destination', 'Last Sync']]
            safe_google_call(output_sheet.update, values=headers, range_name='A1:F1')
            safe_google_call(output_sheet.update, values=rows_to_write, range_name="A2")
            print(f"✅ Success! Updated {len(rows_to_write)} records.")

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    main()